In [1]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
import google.generativeai as genai

In [3]:
load_dotenv(override=True)
api_key = os.getenv("GEM_API_KEY")
genai.configure(api_key=api_key)
MODEL = 'gemini-1.5-flash'

In [53]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    def __init__(self,url):
        """
        Initialize a Website object by fetching and parsing the given webpage URL.

        Args:
            url (str): The full URL of the webpage to retrieve and process.

        Attributes:
            url (str): The original webpage URL.
            body (bytes): The raw HTML content of the webpage.
            title (str): The title of the webpage, or "No Title Found" if unavailable.
            text (str): The main textual content of the webpage body, 
                excluding <script>, <style>, <img>, and <input> elements.
            links (list[str]): A list of all non-empty hyperlink URLs found in the webpage.

        Process:
            - Sends a GET request to fetch the webpage content.
            - Parses the HTML using BeautifulSoup.
            - Extracts the title, if available.
            - Removes irrelevant elements (script, style, images, inputs) from the body.
            - Extracts and stores plain text from the cleaned body.
            - Collects all hyperlink references (<a href="...">) into a list.
        """
        self.url = url
        response = requests.get(url,headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No Title Found"

        if soup.body:
            for irrelevant in soup.body(["script","style","img","input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator ="\n",strip = True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_content(self):
        """
        Retrieve the title and main textual content of the webpage in a formatted string.

        Returns:
            str: A string containing:
            - The webpage title on a new line after "Webpage Title:"
            - The extracted textual content of the webpage body on a new line after "Webpage Contents:"
            - Two newlines at the end for separation.

        Note:
            The textual content excludes script, style, image, and input elements.
        """
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"


In [33]:
ed = Website("https://www.python.org")
ed.links

['#content',
 '#python-network',
 '/',
 'https://www.python.org/psf/',
 'https://docs.python.org',
 'https://pypi.org/',
 '/jobs/',
 '/community/',
 '#top',
 '/',
 'https://psfmember.org/civicrm/contribute/transact?reset=1&id=2',
 '#site-map',
 '#',
 'javascript:;',
 'javascript:;',
 'javascript:;',
 '#',
 'https://www.linkedin.com/company/python-software-foundation/',
 'https://fosstodon.org/@ThePSF',
 '/community/irc/',
 'https://twitter.com/ThePSF',
 '/about/',
 '/about/apps/',
 '/about/quotes/',
 '/about/gettingstarted/',
 '/about/help/',
 'http://brochure.getpython.info/',
 '/downloads/',
 '/downloads/',
 '/downloads/source/',
 '/downloads/windows/',
 '/downloads/macos/',
 '/download/other/',
 'https://docs.python.org/3/license.html',
 '/download/alternatives',
 '/doc/',
 '/doc/',
 '/doc/av',
 'https://wiki.python.org/moin/BeginnersGuide',
 'https://devguide.python.org/',
 'https://docs.python.org/faq/',
 'http://wiki.python.org/moin/Languages',
 'https://peps.python.org',
 'https

In [22]:
def get_links_user_prompt(website):
    """
    Generate a formatted prompt for an AI model to identify relevant company brochure links from a given website.

    The function constructs a textual prompt that lists all extracted links from the specified website
    and instructs the model to return only relevant links in JSON format. Irrelevant links such as 
    Terms of Service, Privacy Policy, or email links should be excluded.

    Args:
        website (Website): An instance of the Website class containing the website's URL and extracted links.

    Returns:
        str: A formatted string containing the website URL, instructions for filtering, 
             and a list of links for the AI to evaluate.
    """
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [23]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://www.python.org - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
#content
#python-network
/
https://www.python.org/psf/
https://docs.python.org
https://pypi.org/
/jobs/
/community/
#top
/
https://psfmember.org/civicrm/contribute/transact?reset=1&id=2
#site-map
#
javascript:;
javascript:;
javascript:;
#
https://www.linkedin.com/company/python-software-foundation/
https://fosstodon.org/@ThePSF
/community/irc/
https://twitter.com/ThePSF
/about/
/about/apps/
/about/quotes/
/about/gettingstarted/
/about/help/
http://brochure.getpython.info/
/downloads/
/downloads/
/downloads/source/
/downloads/windows/
/downloads/macos/
/download/other/
https://docs.python.org/3/license.html
/download/alternatives
/doc/
/doc/
/doc/av
https://wiki.python.org/moin/BeginnersGuide
htt

In [45]:
link_system_prompt = """You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [46]:
def get_links(url):
    """
    Extracts links from the given website URL using Google's Gemini model.

    Args:
        url (str): The website URL to extract links from.

    Returns:
        dict: A JSON object containing extracted links and their details.
    """
    website = Website(url)

    model = genai.GenerativeModel(MODEL)
    response = model.generate_content(
        [
            {"role": "user", "parts": [link_system_prompt + "\n" + get_links_user_prompt(website)]}
        ],
        generation_config={"response_mime_type": "application/json"}
    )
    result = response.text  
    return json.loads(result)

In [47]:
huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/openai/gpt-oss-120b',
 '/openai/gpt-oss-20b',
 '/Qwen/Qwen-Image',
 '/rednote-hilab/dots.ocr',
 '/KittenML/kitten-tts-nano-0.1',
 '/models',
 '/spaces/Qwen/Qwen-Image',
 '/spaces/enzostvs/deepsite',
 '/spaces/black-forest-labs/FLUX.1-Krea-dev',
 '/spaces/amd/gpt-oss-120b-chatbot',
 '/spaces/ZhengGeng/OnePoseviaGen',
 '/spaces',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/jxm/gpt-oss20b-samples',
 '/datasets/HuggingFaceH4/Multilingual-Thinking',
 '/datasets/openai/BrowseCompLongContext',
 '/datasets/miromind-ai/MiroVerse-v0.1',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Writer',
 '/docs/transformers',
 '/docs

In [48]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

In [54]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_content()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_content()
    return result

In [55]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
openai/gpt-oss-120b
Updated
3 days ago
•
490k
•
3.22k
openai/gpt-oss-20b
Updated
3 days ago
•
2.37M
•
2.79k
Qwen/Qwen-Image
Updated
6 days ago
•
69.7k
•
1.49k
rednote-hilab/dots.ocr
Updated
about 2 hours ago
•
17.9k
•
609
KittenML/kitten-tts-nano-0.1
Updated
7 days ago
•
33.7k
•
399
Browse 1M+ models
Spaces
Running
on
Zero
463
463
Qwen Image
🖼
Generate images from text prompts


In [56]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

In [58]:
def get_brochure_user_prompt(company_name, url):
    """
        Build a prompt string for generating a short company brochure in markdown format.

        This function takes a company name and its website URL, retrieves the landing 
        page content and other relevant page details using `get_all_details()`, and 
        formats this information into a prompt suitable for passing to a language model.

        The prompt includes:
        - The company name
        - Instructions to create a markdown-formatted brochure
        - Website content retrieved from the given URL

        The final prompt is truncated to a maximum length of 5,000 characters.

        Args:
            company_name (str): The name of the company.
            url (str): The company's website URL.

        Returns:
            str: A formatted prompt string for brochure generation.
    """
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [59]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nopenai/gpt-oss-120b\nUpdated\n4 days ago\n•\n490k\n•\n3.23k\nopenai/gpt-oss-20b\nUpdated\n4 days ago\n•\n2.37M\n•\n2.81k\nQwen/Qwen-Image\nUpdated\n6 days ago\n•\n69.7k\n•\n1.49k\nrednote-hilab/dots.ocr\nUpdated\nabout 6 hours ago\n•\n17.9k\n•\n613\nKittenML/kitten-tts-nano-0.1\nUpdated\n7 days ago\n•\n33.7k\n•\n400\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\n467\n467\nQwen Image\n🖼\nGe

In [71]:
def stream_brochure(company_name, url):
    """
    Streams an AI-generated brochure for a given company and website.

    Args:
        company_name (str): The name of the company.
        url (str): The company's website URL.

    Returns:
        None
    """
    # Combine your system instructions into the prompt
    prompt = system_prompt + "\n\n" + get_brochure_user_prompt(company_name, url)

    model = genai.GenerativeModel("gemini-1.5-flash")

    # Stream the content directly
    stream = model.generate_content(
        [{"role": "user", "parts": [prompt]}],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        if chunk.candidates and chunk.candidates[0].content.parts:
            text_part = chunk.candidates[0].content.parts[0].text
            response += text_part
            response = response.replace("```", "").replace("markdown", "")
            display_handle.update(Markdown(response))


In [72]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}


# Hugging Face: The AI Community Building the Future

**A Brochure for Prospective Customers, Investors, and Recruits**

**(Image:  A visually appealing graphic incorporating the Hugging Face logo and imagery representing AI collaboration, diverse users, and cutting-edge technology.)**

**Introduction:**

Hugging Face is the leading platform for the machine learning community, fostering collaboration and innovation in the development and deployment of AI. We provide a central hub for sharing models, datasets, and applications, accelerating the progress of AI research and development for individuals, organizations, and enterprises worldwide.

**For Customers:**

* **Access a vast ecosystem:** Explore and leverage over 1 million models and 250,000+ datasets, spanning text, image, video, audio, and 3D modalities.  Find pre-trained models ready for immediate use or customize existing models to your specific needs.
* **Deploy and manage easily:** Host and collaborate on unlimited public models, datasets, and applications with ease.  Our Spaces platform allows you to deploy applications with minimal setup.
* **Accelerate your ML workflow:** Utilize our optimized inference endpoints and GPU computing resources for faster model deployment and improved performance (pricing starts at $0.60/hour for GPU).
* **Join a vibrant community:** Connect with fellow AI enthusiasts, experts, and researchers, sharing knowledge and contributing to the future of AI.

**For Investors:**

* **Market leadership:** Hugging Face is the undisputed leader in AI collaboration, hosting a massive and growing repository of models and datasets.
* **Strong community engagement:**  Our vibrant community of over 50,000 organizations drives continuous innovation and platform growth.  Many industry giants, including Google, Amazon, Microsoft, Meta, and Intel are already leveraging our platform.
* **Scalable business model:** Our platform offers both free and paid services, catering to individuals, research institutions, and enterprises of all sizes.  Our enterprise solutions provide enhanced security and support for large organizations (starting at $20/user/month).
* **Significant future potential:**  As AI continues its rapid expansion, Hugging Face is uniquely positioned to benefit from the increasing demand for collaborative AI development and deployment tools.


**For Recruits:**

* **Be part of something big:** Join a passionate and collaborative team driving the future of artificial intelligence.
* **Cutting-edge technology:** Work with the latest AI technologies, contributing to groundbreaking research and impactful applications.
* **Diverse and inclusive culture:** Hugging Face fosters a diverse and inclusive environment where everyone feels valued and empowered to contribute their unique talents. (Note:  Further detail on specific company culture aspects should be added if available from other sources).
* **Career growth opportunities:**  We offer exciting career paths for engineers, researchers, data scientists, and other professionals in the AI field. (Note:  Include a list of current job openings or a link to the careers page if available).


**Contact:**

For more information, visit our website at [HuggingFace Website Address] or contact us at [Hugging Face Contact Information].

**(Image:  A QR code linking to the Hugging Face website.)**
